## Structure
- load wav file paths, save as a list
- save path of audio to each block 
- for each block load wav file 
- extract envelope
- extract acoustic onset
- extract phoneme onset
- save meta data
- save the file

In [ ]:
from pathlib import Path
import re
import warnings
from matplotlib import pyplot

import numpy as np
import pandas as pd
import textgrid as tg

import eelbrain
from eelbrain import *
import librosa

## User setting

In [ ]:
MAINDIR = Path(r"C:/projects/emo_EEG")
CORPUS_DIR = MAINDIR / "emo_audio" / "mfa_corpus"
CONCAT_AUDIO_DIR = MAINDIR / "emo_audio" / "random_cont"
STIMULI_AUDIO_DIR = MAINDIR / "stimuli"
EMO_ORDER_DIR = STIMULI_AUDIO_DIR / "orders"
OUTPUT_DIR = MAINDIR / "data_pipeline" / "mTRF" / "eelbrain"/ "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTOR_DIR = MAINDIR / "data_pipeline" / "mTRF" / "eelbrain"/ "predictors"
UNIQUE_PHONEME_PATH = CORPUS_DIR / "unique_phones.txt"

N_ORDER = 1
TARGET_FS = 128
ISI = 0.5
LOW_FREQUENCY = 1
HIGH_FREQUENCY = 15

ORDER_FILE = EMO_ORDER_DIR / f"order{N_ORDER}.xlsx"
assert ORDER_FILE.exists(), f"Order file not found: {ORDER_FILE}"

print(f"Using order file: {ORDER_FILE}")
print(f"Output directory: {OUTPUT_DIR}")

info_dict = {
    "target_fs": TARGET_FS,
    "isi": ISI
}

## Helpers

In [ ]:
def parse_block_filename(audio_path_value: str) -> dict:
    """
    Parse filenames like: Gender_Style_Emotion_..._scaled.wav
    and return metadata used downstream.
    """
    audio_path_value = str(audio_path_value)
    orig_name = Path(audio_path_value).name           # e.g., Fem_ADS_hap_xxx_scaled.wav
    stem = Path(orig_name).stem                       # e.g., Fem_ADS_hap_xxx_scaled
    clean_stem = re.sub(r"_scaled$", "", stem)       # remove trailing _scaled only
    parts = clean_stem.split("_")

    if len(parts) < 3:
        raise ValueError(f"Could not parse gender/style/emotion from: {orig_name}")

    return {
        "audio_name_original": orig_name,
        "audio_stem_original": stem,
        "audio_stem_clean": clean_stem,
        "gender": parts[0],
        "speech_style": parts[1],
        "emotion": parts[2],
    } 

def load_unique_phonemes(path) -> list[str]:
    path = Path(path) # convert str -> Path if needed
    phonemes = [line.strip().replace("\r", "") for line in path.read_text(encoding="utf-8").splitlines()]
    phonemes = [p for p in phonemes if p]
    return phonemes

def get_sentence_order_from_table(table_path, target_cont_audio_name):
    """
    Python equivalent of the MATLAB function getSentenceOrderFromTable().

    Parameters
    ----------
    table_path : str or Path
        Path to the .xlsx or .csv table.
    target_cont_audio_name : str
        Target concatenated audio filename to search in the 'output_filename' column.

    Returns
    -------
    order : list[str]
        List of processed sentence names, keeping only the first 4 underscore-separated parts.
        Example: '54_06_hap_f_xxx.wav' -> '54_06_hap_f'
    """

    table_path = Path(table_path)

    # check file exists
    if not table_path.is_file():
        raise FileNotFoundError(f"File does not exist: {table_path}")

    # read table
    if table_path.suffix.lower() == ".xlsx":
        order_tab = pd.read_excel(table_path)
    elif table_path.suffix.lower() == ".csv":
        order_tab = pd.read_csv(table_path)
    else:
        raise ValueError(f"Unsupported file type: {table_path.suffix}")

    # check required columns
    required_cols = ["output_filename", "file_order"]
    for col in required_cols:
        if col not in order_tab.columns:
            raise KeyError(f"Missing required column: {col}")

    # locate target row
    matches = order_tab["output_filename"].astype(str) == str(target_cont_audio_name)
    if not matches.any():
        raise ValueError(f"Target audio file not found in table: {target_cont_audio_name}")

    row_idx = matches.idxmax()
    target_order_list = str(order_tab.loc[row_idx, "file_order"])

    # split by ';' and trim whitespace
    target_order_list = [x.strip() for x in target_order_list.split(";") if x.strip()]

    # keep only first 4 underscore-separated parts
    processed_order = []
    for current_audio_name in target_order_list:
        name_parts = current_audio_name.split("_")
        core_info = "_".join(name_parts[:4])
        processed_order.append(core_info)

    return processed_order

def save_predictors(block_info, predictor_keys, base_data, predictor_dir, n_order,
                    info_dict=None, dataset_prefix="emo_pilot", file_tag=None, show_summary = False):
    """
    Save one or more predictors into a single Eelbrain dataset.

    Parameters
    ----------
    block_info : list of dict
        Each dict contains extracted predictors.
    predictor_keys : str or list of str
        Predictor name(s) to save.
    base_data : dict
        Precomputed shared dataset columns, e.g. stim_idx, filename, gender.
    predictor_dir : Path or str
        Output directory.
    n_order : int
        Order number used in output filename.
    info_dict : dict or None
        Optional metadata to store in ds.info.
    dataset_prefix : str
        Prefix used in dataset name and filename.
    file_tag : str or None
        Optional custom tag for output filename.

    Returns
    -------
    ds : eelbrain.Dataset
        Saved dataset.
    out_path : Path
        Output file path.
    """
    predictor_dir = Path(predictor_dir)

    if isinstance(predictor_keys, str):
        predictor_keys = [predictor_keys]
    else:
        predictor_keys = list(predictor_keys)

    if not predictor_keys:
        raise ValueError("predictor_keys is empty.")

    # check predictors exist in all blocks
    for key in predictor_keys:
        missing_idx = [i for i, block in enumerate(block_info) if key not in block]
        if missing_idx:
            raise ValueError(f"Predictor '{key}' is missing from blocks: {missing_idx}")

    ds = Dataset(name=f"{dataset_prefix}_{'_'.join(predictor_keys)}")

    # add shared columns
    for col_name, col_value in base_data.items():
        ds[col_name] = col_value

    # add predictors
    for key in predictor_keys:
        ds[key] = [block[key] for block in block_info]

    # add optional metadata
    if info_dict is not None:
        for k, v in info_dict.items():
            ds.info[k] = v

    ds.info["predictor_names"] = predictor_keys

    # output filename
    if file_tag is None:
        file_tag = "_".join(predictor_keys)

    out_path = predictor_dir / f"{dataset_prefix}_order{n_order}_{file_tag}.pickle"
    eelbrain.save.pickle(ds, out_path)

    print(f"Saved predictor dataset to: {out_path}")
    
    if show_summary:
        print(ds)
        print(ds.summary())

    return ds, out_path

def get_track_phoneme_info(textgrid_path):
        
        if not textgrid_path.exists():
            raise FileNotFoundError(f"{textgrid_path} not found")

        tg_obj = tg.TextGrid.fromFile(textgrid_path)

        # Get the "phones" tier
        if "phones" in tg_obj.getNames():
            phones_tier = tg_obj.getFirst("phones")
        else:
            raise ValueError(f"'phones' tier not found in {textgrid_path}")

        t1 = np.array([interval.minTime for interval in phones_tier])
        t2 = np.array([interval.maxTime for interval in phones_tier])
        # remove trailing stress digits like AH0 -> AH, IH1 -> IH, AH1 -> AH
        labels = np.array([
            re.sub(r'\d+$', '', interval.mark)
            for interval in phones_tier
        ])

        tmin = t1[0]
        tmax = t2[-1]

        stim_name = textgrid_path.name
        return {'stim_name': stim_name, 'tmin': tmin, 'tmax': tmax, 't1': t1, 't2': t2, 'labels': labels}

## Step 1: load all selected wav files

In [ ]:
order_df = pd.read_excel(ORDER_FILE)
if "audio_path" not in order_df.columns or "stim_idx" not in order_df.columns:
    raise ValueError("Expected columns `audio_path` and `stim_idx` in the order file.")

order_df = order_df.loc[order_df["stim_idx"].notna() & (order_df["stim_idx"] > 0)].copy()
order_df = order_df.sort_values("stim_idx").reset_index(drop=True)

block_info = []

for _, row in order_df.iterrows():
    meta = parse_block_filename(row["audio_path"])
    audio_file_path = STIMULI_AUDIO_DIR / meta["audio_name_original"]

    if not audio_file_path.exists():
        raise FileNotFoundError(f"Missing audio file: {audio_file_path}")

    # load wav directly as Eelbrain NDVar
    wav_nd = eelbrain.load.wav(audio_file_path, name=meta["audio_stem_clean"])

    # if a file has multiple channels, average to mono
    # per Eelbrain docs, multichannel wav dims are (time, channel)
    if len(wav_nd.dims) == 2:
        wav_nd = wav_nd.mean("channel")

    fs_audio = wav_nd.info["samplingrate"]

    block_info.append({
        **meta,
        "stim_idx": int(row["stim_idx"]),
        "audio_file_path": audio_file_path,
        "wav": wav_nd,
        "fs_audio": fs_audio,
    })

print(f"Loaded {len(block_info)} wav files.")
print("First file:", block_info[0]["audio_name_original"])

In [ ]:
# shared metadata columns
base_data = {
    "stim_idx": Var([block["stim_idx"] for block in block_info], name="stim_idx"),
    "filename": Factor([block["audio_name_original"] for block in block_info], name="filename"),
    "gender": Factor([block["gender"] for block in block_info], name="gender"),
    "speech_style": Factor([block["speech_style"] for block in block_info], name="speech_style"),
    "emotion": Factor([block["emotion"] for block in block_info], name="emotion"),
}

## Step 2: derive the envelope

In [ ]:
for block in block_info:
    wav_nd = block["wav"]
    stim_name = wav_nd.name

    # Hilbert envelope from Eelbrain NDVar, then resample to target fs
    envelope = wav_nd.envelope(name=f"{stim_name}_envelope")
    # Filter the envelope with the same parameters as the EEG data
    envelope = eelbrain.filter_data(envelope, LOW_FREQUENCY, HIGH_FREQUENCY, pad='reflect')
    envelope = eelbrain.resample(envelope, TARGET_FS)

    block["envelope"] = envelope
    block["duration"] = float(envelope.time.tstop)

print("Envelope extraction done.")
print(block_info[0]["envelope"])

if all("duration" in block for block in block_info):
    base_data["duration"] = Var([block["duration"] for block in block_info], name="duration")

save_predictors(
    block_info=block_info,
    predictor_keys='envelope',
    base_data=base_data,
    predictor_dir=PREDICTOR_DIR,
    n_order=N_ORDER,
    info_dict=info_dict
)

## Step3: derive the acoustic onsets

In [ ]:
for block in block_info:
    wav_nd = block["wav"]
    stim_name = wav_nd.name
    if "envelope" not in block:
        raise RuntimeError("Run the envelope cell first.")

    envelope = block["envelope"]
    onset = envelope.diff('time', name=f'{stim_name}_onset').clip(0)

    onset_max = float(onset.max())
    env_max = float(envelope.max())

    if onset_max > 0:
        onset *= env_max / onset_max
    else:
        warnings.warn(f"Onset max is zero for {block['audio_name_original']}; leaving onset unscaled.")

    block["acoustic_onset"] = onset

print("Acoustic onset extraction done.")
print(block_info[0]["acoustic_onset"])

save_predictors(
    block_info=block_info,
    predictor_keys='acoustic_onset',
    base_data=base_data,
    predictor_dir=PREDICTOR_DIR,
    n_order=N_ORDER,
    info_dict=info_dict
)

In [ ]:
# Visualize the first 5 seconds
p = eelbrain.plot.UTS([block_info[0]["wav"], block_info[0]["envelope"] * 2], axh=2, w=10, columns=1, xlim=10)
# Add y=0 as reference
p.add_hline(0, zorder=0)

#fig, axes = pyplot.subplots(2, 1, sharex=True, figsize=(10, 4))
p = eelbrain.plot.UTS([block_info[0]["envelope"] , block_info[0]["acoustic_onset"] ], axh=2, w=10, columns=1, xlim=10)

## Step 4: derive the gammatone vector

In [ ]:
for block in block_info:
    wav_nd = block["wav"]
    stim_name = wav_nd.name
    # Apply a gammatone filterbank, producing a high resolution spectrogram
    gt = eelbrain.gammatone_bank(
        wav_nd, 80, 10000, 128, location='left', tstep = 0.001
        )
    # Apply a log transform to approximate peripheral auditory processing
    gt_log = (gt + 1).log()
    # Apply the edge detector model to generate an acoustic onset spectrogram
    gt_on = eelbrain.edge_detector(gt_log, c=30)
    # Create and save 8 band versions of the two predictors (binning the frequency axis into 8 bands)
    gt_8 = gt_log.bin(nbins=8, func='sum', dim='frequency')
    gt_on_8 = gt_on.bin(nbins=8, func='sum', dim='frequency')
    # Create gammatone spectrograms with linear scale, only 8 bin versions
    gt_ln_8 = gt.bin(nbins=8, func='sum', dim='frequency')
    # Powerlaw scale
    gt_pow = gt ** 0.6
    gt_pow_8 = gt_pow.bin(nbins=8, func='sum', dim='frequency')

    # Resample to EEG sampling rate (128 Hz -> 1/128 s)
    gt_rs = eelbrain.resample(gt, TARGET_FS)
    gt_log_sum_rs = eelbrain.resample(gt_log.sum('frequency'), TARGET_FS)
    gt_on_sum_rs = eelbrain.resample(gt_on.sum('frequency'), TARGET_FS)
    gt_ln_8_rs = eelbrain.resample(gt_ln_8, TARGET_FS)
    gt_8_rs = eelbrain.resample(gt_8, TARGET_FS)
    gt_on_8_rs = eelbrain.resample(gt_on_8, TARGET_FS)
    gt_pow_8_rs = eelbrain.resample(gt_pow_8, TARGET_FS)

    block["gammatone"] = gt_rs.copy(name = "gammatone")
    block["gammatone_1"] = gt_log_sum_rs.copy(name = "gammatone_1")
    block["gammatone_on_1"] = gt_on_sum_rs.copy(name = "gammatone_on_1")
    block["gammatone_ln_8"] = gt_ln_8_rs.copy(name = "gammatone_ln_8")
    block["gammatone_8"] = gt_8_rs.copy(name = "gammatone_8")
    block["gammatone_on_8"] = gt_on_8_rs.copy(name = "gammatone_on_8")
    block["gammatone_pwr_8"] = gt_pow_8_rs.copy(name = "gammatone_pwr_8")

print("Gammatone features extraction done.")
print(block_info[0]["gammatone_8"])

gammatone_keys = [
    "gammatone",
    "gammatone_1",
    "gammatone_on_1",
    "gammatone_ln_8",
    "gammatone_8",
    "gammatone_on_8",
    "gammatone_pwr_8",
]

for key in gammatone_keys:
    save_predictors(
    block_info=block_info,
    predictor_keys= key,
    base_data=base_data,
    predictor_dir=PREDICTOR_DIR,
    n_order=N_ORDER,
    info_dict=info_dict
    )

In [ ]:
# choose one block to plot
block_idx = 0
b = block_info[block_idx]

# get predictors
gt_8 = b["gammatone_8"]
gt_on_8 = b["gammatone_on_8"]

# summed envelopes
env_8 = gt_8.sum("frequency")
env_on_8 = gt_on_8.sum("frequency")

# optional time window
xlim = (0, 3)   # example: (0, 3.0)

if xlim is not None:
    gt_8 = gt_8.sub(time=xlim)
    gt_on_8 = gt_on_8.sub(time=xlim)
    env_8 = env_8.sub(time=xlim)
    env_on_8 = env_on_8.sub(time=xlim)

# helper: rescale 1D envelope so it can be overlaid on spectrogram
def rescale_for_overlay(nd, y_offset=0.5, y_height=7.0):
    y = np.asarray(nd.x, dtype=float).copy()
    y -= y.min()
    ymax = y.max()
    if ymax > 0:
        y *= y_height / ymax
    y += y_offset
    return nd.time.times, y

# build figure
fig = pyplot.figure(figsize=(10, 8))
gs = fig.add_gridspec(2, 1, hspace=0.45)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0])

# plot spectrograms
eelbrain.plot.Array(
    gt_8,
    axes=[ax1],
    axtitle=False,
    xticklabels='bottom',
    yticklabels=False,
    xlim=xlim,
)
eelbrain.plot.Array(
    gt_on_8,
    axes=[ax2],
    axtitle=False,
    xticklabels='bottom',
    yticklabels=False,
    xlim=xlim,
)

# overlay summed envelopes
t1, y1 = rescale_for_overlay(env_8)
ax1.plot(t1, y1, linewidth=2)
ax1.set_title("gammatone-8 with summed envelope", loc="left")

t2, y2 = rescale_for_overlay(env_on_8)
ax2.plot(t2, y2, linewidth=2)
ax2.set_title("gammatone-on-8 with summed envelope", loc="left")

fig.suptitle(f'Block {block_idx}: {b["audio_name_original"]}', y=0.98)
pyplot.show()

## Step 5: Derive individual phoneme predictor

In [ ]:
phonemes_to_include = load_unique_phonemes(UNIQUE_PHONEME_PATH)
#phonemes_to_include = ['DH', 'T', 'IH', 'N', 'R', 'OW', 'Z', 'D', 'AW',
#                 'L', 'ER', 'K', 'M', 'S', 'HH', 'AA', 'SH', 'EY',
#                 'G', 'W', 'P', 'AY', 'AO', 'F', 'UW', 'AE']
ph2idx = {ph: i for i, ph in enumerate(phonemes_to_include)}
phoneme_dim = Categorial('phoneme', phonemes_to_include)
n_phonemes = len(phonemes_to_include)
print("Unique phonemes:", len(phonemes_to_include))

for block in block_info:
    current_emotion = block['emotion']
    current_speech_style = block['speech_style']
    current_gender = block['gender']

    sent_order_tab_folder = CONCAT_AUDIO_DIR / current_speech_style / f'{current_gender}_{current_speech_style}'
    sent_order_tab_name = f'{current_gender}_{current_speech_style}_random_cont_order.csv'
    sent_order_tab_path = sent_order_tab_folder / sent_order_tab_name

    # resample block waveform to target fs for timing grid
    wav_nd = block["wav"]
    stim_name = wav_nd.name
    wav_ds = wav_nd.bin(1/TARGET_FS, dim='time', label='start')
    time_arr = wav_ds.time.times
    time_dim = wav_ds.get_dim('time')

    n_times = len(time_arr)

    # predictor shape: phoneme x times
    predictor = np.zeros((n_phonemes, n_times), dtype=float)
    
    if sent_order_tab_path.exists():
        sent_order_list = get_sentence_order_from_table(sent_order_tab_path, f'{block['audio_stem_clean']}.wav')
    else:
        Warning(f'Sentence order table not found for {sent_order_tab_name}')
        continue

    block_phonemes = []
    block_label_onset_times = []
    block_label_onset_indexes = []

    start_t = 0 
    for sentence_name in sent_order_list:
        sent_txt_name = f'{sentence_name}_cleaned.TextGrid'
        sent_txt_folder = CORPUS_DIR / current_speech_style / f'{current_gender}_{current_speech_style}'/ 'aligned'
        sent_txt_path = sent_txt_folder / sent_txt_name

        phoneme_info = get_track_phoneme_info(sent_txt_path)
        onset_times = phoneme_info['t1'] + start_t
        offset_times = phoneme_info['t2'] + start_t

        sent_audio_name = f'{sentence_name}_cleaned.wav'
        sent_audio_path = CORPUS_DIR / current_speech_style / f'{current_gender}_{current_speech_style}'/ sent_audio_name
        sent_wav = eelbrain.load.wav(sent_audio_path, name = sentence_name)
        sent_wav_ds = sent_wav.bin(1/TARGET_FS, dim='time', label='start')
        sent_dur = sent_wav_ds.time.tstop

        for idx, phoneme in enumerate(phoneme_info['labels']):
            if not phoneme in phonemes_to_include:
                continue
            onset_t = onset_times[idx] 
            onset_idx = np.argmin(np.abs(time_arr - onset_t))

            if not (0 <= onset_idx < n_times):
                continue

            ph_idx = ph2idx[phoneme]
            predictor[ph_idx, onset_idx] = 1.0

            block_phonemes.append(phoneme)
            block_label_onset_times.append(onset_t)
            block_label_onset_indexes.append(onset_idx)

        start_t = start_t + sent_dur + ISI

    # keep only phonemes that occurred at least once in this block
    keep_mask = predictor.sum(axis=1) > 0
    predictor = predictor[keep_mask, :]
    kept_phonemes = [ph for ph, keep in zip(phonemes_to_include, keep_mask) if keep]
    phoneme_dim_block = Categorial('phoneme', kept_phonemes)

    predictor_nd = NDVar(
        predictor,
        (phoneme_dim_block, time_dim),
        name="phoneme_onset"
    )

    block['phoneme_onset'] = predictor_nd
    block['duration'] = float(wav_ds.time.tstop)
    block['phonemes'] = block_phonemes
    block['phoneme_onset_times'] = block_label_onset_times
    block['phoneme_onset_indexes'] = block_label_onset_indexes
    block['phonemes_in_block'] = kept_phonemes

print("Phoneme-onset predictor extraction done.")
print(block_info[0]['phoneme_onset'])

if all("duration" in block for block in block_info):
    base_data["duration"] = Var([block["duration"] for block in block_info], name="duration")

save_predictors(
    block_info=block_info,
    predictor_keys='phoneme_onset',
    base_data=base_data,
    predictor_dir=PREDICTOR_DIR,
    n_order=N_ORDER,
    info_dict=info_dict
)

## Step 6: MFCC

In [ ]:
n_MFCC = 13

for block in block_info:
    wav_nd = block["wav"]
    sf_wav_nd = 1 / wav_nd.time.tstep
    stim_name = wav_nd.name
    
    y = np.asarray(wav_nd.x, dtype=np.float64).squeeze()
    # Choose hop_length so MFCC frame rate is close to TARGET_FS
    hop_length = int(round(sf_wav_nd / TARGET_FS))
    hop_length = max(hop_length, 1)

    # Compute MFCCs
    # Output shape: (n_MFCC, n_frames)
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=int(round(sf_wav_nd)),
        n_mfcc=n_MFCC,
        hop_length=hop_length
    )

    # MFCC frame rate after framing
    mfcc_fs = sf_wav_nd / hop_length

    # Resample MFCC time axis exactly to TARGET_FS if needed
    if not np.isclose(mfcc_fs, TARGET_FS):
        mfcc = librosa.resample(
            mfcc,
            orig_sr=mfcc_fs,
            target_sr=TARGET_FS,
            axis=1
        )

    # Create Eelbrain dimensions
    mfcc_dim = Scalar("mfcc", np.arange(1, n_MFCC + 1))
    time_dim = UTS(0, 1 / TARGET_FS, mfcc.shape[1])

    # Save as NDVar: (mfcc, time)
    mfcc_nd = NDVar(
        mfcc,
        dims=(mfcc_dim, time_dim),
        name="mfcc"
    )
    
    block['MFCCs'] = mfcc_nd

print("MFCCs predictor extraction done.")
print(block_info[0]['MFCCs'])

save_predictors(
    block_info=block_info,
    predictor_keys='MFCCs',
    base_data=base_data,
    predictor_dir=PREDICTOR_DIR,
    n_order=N_ORDER,
    info_dict=info_dict
)

In [ ]:
fig, ax = pyplot.subplots()

img = librosa.display.specshow(
    mfcc,
    x_axis='time',
    ax=ax
)

fig.colorbar(img, ax=ax)
ax.set_title('MFCC')

pyplot.show()

In [ ]:
print(wav_ds)